# InstaSearch Training Workflow

This notebook demonstrates how to load the proteomics dataset, prepare the dual-encoder model, and run a training epoch using the repository code in `src/`. It also shows how to execute the full `train_script.py` command-line workflow.

In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd()))
sys.path.insert(0, os.path.join(repo_root, 'src'))

print('Repo root:', repo_root)
print('Python executable:', sys.executable)

In [ ]:
import torch
from datasets import load_dataset

from data.preprocess import preprocess_dataset
from data.dataset import SpectraPeptideDataset
from models.peptide_encoder import PeptideEncoder
from models.spectrum_encoder import SpectrumEncoder
from training.loss import CLIPContrastiveLoss
from training.train import train_epoch, validate
from utils.constants import DEVICE
from utils.visualization import plot_metrics, plot_similarity_matrix, plot_embeddings

print('Torch version:', torch.__version__)
print('Device:', DEVICE)

In [ ]:
dataset_name = 'InstaDeepAI/ms_ninespecies_benchmark'
split = 'train[:2000]'

print('Loading dataset:', dataset_name, split)
raw_ds = load_dataset(dataset_name, split=split)
df = raw_ds.to_pandas()

specs, peps, pres = preprocess_dataset(df)
print('Preprocessed examples:', len(specs))

dataset = SpectraPeptideDataset(specs, peps, pres)
print('Dataset length:', len(dataset))

In [ ]:
batch_size = 64
train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

model_spec = SpectrumEncoder(d_model=256, n_heads=4, d_ff=512, n_layers=2, embed_dim=64).to(DEVICE)
model_pep = PeptideEncoder(d_model=256, n_heads=4, d_ff=512, n_layers=2, embed_dim=64).to(DEVICE)
loss_fn = CLIPContrastiveLoss(init_temp=0.1).to(DEVICE)
optimizer = torch.optim.AdamW(list(model_spec.parameters()) + list(model_pep.parameters()) + [loss_fn.log_temp], lr=1e-4, weight_decay=5e-3)

print('Model spec params:', sum(p.numel() for p in model_spec.parameters()))
print('Model peptide params:', sum(p.numel() for p in model_pep.parameters()))

In [ ]:
from torch.utils.data import random_split

# Split the dataset the same way the training script does
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False)

scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
num_epochs = 3

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model_spec, model_pep, train_loader, loss_fn, optimizer, scaler)
    val_loss, val_acc = validate(model_spec, model_pep, val_loader, loss_fn)

    history['loss'].append(train_loss)
    history['acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch {epoch + 1}/{num_epochs} | '
        f'Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}'
    )

## Training Visualization

The notebook now uses the shared visualization utilities from `src/utils/visualization.py`.

In [ ]:
# Plot training and validation metrics using shared repo utilities
plot_metrics(history)

# Visualize a single batch similarity matrix and embeddings if data is available
batch = next(iter(val_loader))
specs_batch, peps_batch, pres_batch = batch
z_spec = model_spec(specs_batch.to(DEVICE), pres_batch.to(DEVICE))
z_pep = model_pep(peps_batch.to(DEVICE))

plot_similarity_matrix(z_spec, z_pep)
plot_embeddings(z_spec, z_pep)

## Run the full training script

The repository also includes `train_script.py`, which implements the full CLI-based training workflow with dataset loading, checkpoint saving, and validation. Run this command from the repository root to train with a larger subset:

```bash
python train_script.py --dataset InstaDeepAI/ms_ninespecies_benchmark --dataset_split train[:20000] --batch_size 128 --num_epochs 10 --output_dir ./checkpoints
```